### Análise Exploratória de Dados (EDA) - Camada Bronze
Este notebook tem como objetivo identificar inconsistências, outliers e padrões nos dados brutos das Emendas Parlamentares para guiar a lógica de transformação da camada Silver.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

TABLE_PATH = "data/01_bronze/emendas_parlamentares/emendasparlamentares"

spark = SparkSession.builder \
    .appName("EDA_Emendas_Parlamentares") \
    .getOrCreate()

df_emendas = spark.read.parquet(TABLE_PATH)

def profiling_emendas_senior(df):
    total_rows = df.count()
    print(f"Total de registros: {total_rows}\n")

    print("--- ANÁLISE DE QUALIDADE POR COLUNA ---")
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            condition = F.col(col_name).isNull() | (F.col(col_name) == "")
        else:
            condition = F.col(col_name).isNull()

        null_count = df.filter(condition).count()
        null_pct = (null_count / total_rows) * 100
        
        distinct_count = df.select(col_name).distinct().count()
        
        print(f"Coluna: {col_name}")
        print(f" > Tipo: {col_type}")
        print(f" > Nulos/Vazios: {null_count} ({null_pct:.2f}%)")
        print(f" > Valores Únicos: {distinct_count}")
        
        print(f" > Top 3 valores:")
        df.groupBy(col_name).count().orderBy(F.desc("count")).show(3, truncate=False)
        print("-" * 30)

profiling_emendas_senior(df_emendas)

Total de registros: 88993

--- ANÁLISE DE QUALIDADE POR COLUNA ---
Coluna: Código da Emenda
 > Tipo: string
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 71127
 > Top 3 valores:
+----------------+-----+
|Código da Emenda|count|
+----------------+-----+
|Sem informação  |17809|
|202320380003    |2    |
|202030410010    |2    |
+----------------+-----+
only showing top 3 rows
------------------------------
Coluna: Ano da Emenda
 > Tipo: int
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 13
 > Top 3 valores:


+-------------+-----+
|Ano da Emenda|count|
+-------------+-----+
|2014         |11154|
|2017         |9633 |
|2020         |8621 |
+-------------+-----+
only showing top 3 rows
------------------------------
Coluna: Tipo de Emenda
 > Tipo: string
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 5
 > Top 3 valores:
+----------------------------------------------------------+-----+
|Tipo de Emenda                                            |count|
+----------------------------------------------------------+-----+
|Emenda Individual - Transferências com Finalidade Definida|75050|
|Emenda Individual - Transferências Especiais              |4430 |
|Emenda de Relator                                         |3892 |
+----------------------------------------------------------+-----+
only showing top 3 rows
------------------------------


Coluna: Código do Autor da Emenda
 > Tipo: string
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 1560
 > Top 3 valores:
+-------------------------+-----+
|Código do Autor da Emenda|count|
+-------------------------+-----+
|S/I                      |15961|
|-99                      |1848 |
|8100                     |331  |
+-------------------------+-----+
only showing top 3 rows
------------------------------
Coluna: Nome do Autor da Emenda
 > Tipo: string
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 1599
 > Top 3 valores:
+-----------------------+-----+
|Nome do Autor da Emenda|count|
+-----------------------+-----+
|Sem informação         |15961|
|RELATOR GERAL          |2179 |
|JANDIRA FEGHALI        |224  |
+-----------------------+-----+
only showing top 3 rows
------------------------------
Coluna: Número da emenda
 > Tipo: string
 > Nulos/Vazios: 0 (0.00%)
 > Valores Únicos: 341
 > Top 3 valores:
+----------------+-----+
|Número da emenda|count|
+----------------+-----+
|S/I    

In [19]:
def profiling_emendas_full_others(df):
    total_rows = df.count()
    print(f"=== ANÁLISE COMPLETA DE NÃO-NUMÉRICOS (Total: {total_rows} linhas) ===\n")

    # Filtramos apenas colunas string
    str_cols = [c for c, t in df.dtypes if t == "string"]

    for col_name in str_cols:
        # Regex: Apenas dígitos de 0 a 9 do início ao fim
        is_numeric = F.col(col_name).rlike("^[0-9]+$")
        
        count_numeric = df.filter(is_numeric).count()
        count_others = total_rows - count_numeric
        
        perc_num = (count_numeric / total_rows) * 100
        perc_others = (count_others / total_rows) * 100

        print(f"COLUNA: {col_name}")
        print(f" > [NUMÉRICO] {count_numeric:>10} ({perc_num:6.2f}%)")
        print(f" > [OUTROS]   {count_others:>10} ({perc_others:6.2f}%)")

        if count_others > 0 and count_numeric > 0:
            print(f" > LISTAGEM COMPLETA DE VALORES NÃO-NUMÉRICOS EM '{col_name}':")
            # .show(n=total_rows) garante que tudo seja impresso
            # truncate=False impede que o Spark coloque "..." em textos longos
            df.filter(~is_numeric) \
              .groupBy(col_name) \
              .count() \
              .orderBy(F.desc("count")) \
              .show(n=total_rows, truncate=False)
        else:
            print(" > (Nenhum valor não-numérico encontrado)")
        
        print("-" * 60)

profiling_emendas_full_others(df_emendas)

=== ANÁLISE COMPLETA DE NÃO-NUMÉRICOS (Total: 88993 linhas) ===

COLUNA: Código da Emenda
 > [NUMÉRICO]      71184 ( 79.99%)
 > [OUTROS]        17809 ( 20.01%)
 > LISTAGEM COMPLETA DE VALORES NÃO-NUMÉRICOS EM 'Código da Emenda':
+----------------+-----+
|Código da Emenda|count|
+----------------+-----+
|Sem informação  |17809|
+----------------+-----+

------------------------------------------------------------
COLUNA: Tipo de Emenda
 > [NUMÉRICO]          0 (  0.00%)
 > [OUTROS]        88993 (100.00%)
 > (Nenhum valor não-numérico encontrado)
------------------------------------------------------------
COLUNA: Código do Autor da Emenda
 > [NUMÉRICO]      71184 ( 79.99%)
 > [OUTROS]        17809 ( 20.01%)
 > LISTAGEM COMPLETA DE VALORES NÃO-NUMÉRICOS EM 'Código do Autor da Emenda':
+-------------------------+-----+
|Código do Autor da Emenda|count|
+-------------------------+-----+
|S/I                      |15961|
|-99                      |1848 |
+-------------------------+-----+

-